---
title: "Random"
format: html
---

<div id="question-bank" style="display: none !important;">

{{< include ../exercises/exercises.qmd >}}

</div>

<div id="exercise-app-root" class="my-4"></div>

```{=html}
<script>
document.addEventListener("DOMContentLoaded", function () {
  const root = document.getElementById('exercise-app-root');
  const sourcePool = document.getElementById('question-bank');

  if (!root || !sourcePool) return;

  // Find all exercise boxes inside the hidden source pool
  const allExercises = Array.from(sourcePool.querySelectorAll('.exercise-box'));

  if (allExercises.length === 0) {
    root.innerHTML = '<div class="alert alert-warning">No exercises with class <code>.exercise-box</code> found in source file.</div>';
    return;
  }

  // Assign stable IDs: uses native Quarto #id first, or content hash fallback
  allExercises.forEach((el) => {
    if (!el.id) {
      const textContent = el.textContent.trim();
      let hash = 0;
      for (let i = 0; i < textContent.length; i++) {
        hash = ((hash << 5) - hash) + textContent.charCodeAt(i);
        hash |= 0;
      }
      el.id = 'exr_hash_' + Math.abs(hash);
    }
  });

  // Inject UI structure dynamically into root
  root.innerHTML = `
    <div class="card p-3 my-3">
      <div class="d-flex align-items-center justify-content-between flex-wrap gap-3">
        <button id="next-exr-btn" class="btn btn-primary">
          ↻ Next
        </button>

        <!-- Filter Dropdown & Counters -->
        <div class="d-flex align-items-center gap-2">
          <select id="filter-select" class="form-select form-select-sm style="width: auto;">
            <option value="all">All questions</option>
            <option value="unrated">❓ Unrated only</option>
            <option value="down">👎 Needs review only</option>
            <option value="up">👍 Good only</option>
          </select>
        </div>
      </div>

      <!-- Stats Pill Bar -->
      <div class="d-flex gap-2 mt-3 pt-2 border-top small text-muted">
        <span>Total: <strong id="cnt-all">0</strong></span> |
        <span>Unrated: <strong id="cnt-unrated" class="text-secondary">0</strong></span> |
        <span>Good: <strong id="cnt-up" class="text-success">0</strong></span> |
        <span>Needs review: <strong id="cnt-down" class="text-danger">0</strong></span>
      </div>
    </div>

    <!-- Active Single Exercise Display -->
    <div id="random-exercise-display"></div>

    <!-- Feedback Bar attached to active exercise -->
    <div id="feedback-bar" class="mt-3 p-2 bg-light border rounded d-flex align-items-center gap-2" style="display:none;">
      <span class="small text-muted me-2">Rate this question:</span>
      <button id="btn-thumbs-up" class="btn btn-outline-success btn-sm">👍 Good</button>
      <button id="btn-thumbs-down" class="btn btn-outline-danger btn-sm">👎 Needs review</button>
    </div>
  `;

  // Get DOM references
  const nextBtn = document.getElementById('next-exr-btn');
  const filterSelect = document.getElementById('filter-select');
  const displayContainer = document.getElementById('random-exercise-display');
  const feedbackBar = document.getElementById('feedback-bar');
  const btnUp = document.getElementById('btn-thumbs-up');
  const btnDown = document.getElementById('btn-thumbs-down');

  // Stats elements
  const cntAll = document.getElementById('cnt-all');
  const cntUnrated = document.getElementById('cnt-unrated');
  const cntUp = document.getElementById('cnt-up');
  const cntDown = document.getElementById('cnt-down');

  let activeExerciseId = null;

  function getVote(id) {
    return localStorage.getItem('vote_' + id);
  }

  // Update real-time counter metrics
  function updateCounters() {
    const total = allExercises.length;
    const upCount = allExercises.filter(el => getVote(el.id) === 'up').length;
    const downCount = allExercises.filter(el => getVote(el.id) === 'down').length;
    const unratedCount = total - (upCount + downCount);

    cntAll.textContent = total;
    cntUnrated.textContent = unratedCount;
    cntUp.textContent = upCount;
    cntDown.textContent = downCount;
  }

  // Get exercise pool matching the active dropdown filter
  function getActivePool() {
    const filter = filterSelect.value;
    if (filter === 'unrated') {
      return allExercises.filter(el => !getVote(el.id));
    } else if (filter === 'down') {
      return allExercises.filter(el => getVote(el.id) === 'down');
    } else if (filter === 'up') {
      return allExercises.filter(el => getVote(el.id) === 'up');
    }
    return allExercises;
  }

  function updateVoteUI() {
    if (!activeExerciseId) return;
    const vote = getVote(activeExerciseId);

    btnUp.className = vote === 'up' ? 'btn btn-success btn-sm' : 'btn btn-outline-success btn-sm';
    btnDown.className = vote === 'down' ? 'btn btn-danger btn-sm' : 'btn btn-outline-danger btn-sm';
  }

  function showRandomExercise() {
    const pool = getActivePool();

    if (pool.length === 0) {
      const filter = filterSelect.value;
      let msg = 'No exercises available.';
      if (filter === 'unrated') msg = 'All exercises have been rated!';
      else if (filter === 'down') msg = 'No exercises flagged with 👎 Needs review!';
      else if (filter === 'up') msg = 'No exercises marked with 👍 Good yet!';

      displayContainer.innerHTML = `<div class="alert alert-info m-0">${msg}</div>`;
      feedbackBar.style.display = 'none';
      activeExerciseId = null;
      return;
    }

    feedbackBar.style.display = 'flex';

    // Pick random question from current pool without immediate repeat
    let chosenElement;
    if (pool.length > 1 && activeExerciseId) {
      const remainingPool = pool.filter(el => el.id !== activeExerciseId);
      const randomIndex = Math.floor(Math.random() * remainingPool.length);
      chosenElement = remainingPool[randomIndex];
    } else {
      chosenElement = pool[0];
    }

    activeExerciseId = chosenElement.id;

    // Clone element to leave source intact
    const clone = chosenElement.cloneNode(true);
    clone.style.display = 'block';

    displayContainer.innerHTML = '';
    displayContainer.appendChild(clone);

    // Re-render MathJax equations
    if (window.MathJax && window.MathJax.typesetPromise) {
      window.MathJax.typesetPromise([displayContainer]);
    }

    updateVoteUI();
  }

  // Handle rating votes
  btnUp.addEventListener('click', function() {
    if (!activeExerciseId) return;
    const currentVote = getVote(activeExerciseId);
    
    if (currentVote === 'up') {
      localStorage.removeItem('vote_' + activeExerciseId);
    } else {
      localStorage.setItem('vote_' + activeExerciseId, 'up');
    }
    
    updateVoteUI();
    updateCounters();
  });

  btnDown.addEventListener('click', function() {
    if (!activeExerciseId) return;
    const currentVote = getVote(activeExerciseId);

    if (currentVote === 'down') {
      localStorage.removeItem('vote_' + activeExerciseId);
    } else {
      localStorage.setItem('vote_' + activeExerciseId, 'down');
    }

    updateVoteUI();
    updateCounters();
  });

  filterSelect.addEventListener('change', showRandomExercise);
  nextBtn.addEventListener('click', showRandomExercise);

  // Initialize
  updateCounters();
  showRandomExercise();
});
</script>